# GEDI Canopy Height Pipeline

This notebook processes NASA GEDI Level 2A HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/`  
**Final output:** `s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary-cville.csv`

## Cell 1 — Install Dependencies

In [5]:
import subprocess, sys

# REMOVED: the strict "botocore==1.43.36" version pin
packages = ["s3fs==2026.6.0", "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All dependencies installed.")

All dependencies installed.


## Cell 2 — Imports

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
Cell In[2], line 7
      4 import re
      5 import urllib.request
----> 7 import boto3
      8 import s3fs
      9 import h5py

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/__init__.py:18
     15 from logging import NullHandler
     17 from boto3.compat import _warn_deprecated_python
---> 18 from boto3.session import Session
     20 __author__ = 'Amazon Web Services'
     21 __version__ = '1.43.36'

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/session.py:29
     26 import boto3.utils
     27 from boto3.exceptions import ResourceNotExistsError, UnknownAPIVersionError
---> 29 from .resources.factory import ResourceFactory
     32 class Session:
     33     """
     34     A session stores configuration state and allows you to create service
     35     clients and resources.
   (...)
     52     :param aws_account_id: AWS account ID
     53     """

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/resources/factory.py:17
     14 import logging
     15 from functools import partial
---> 17 from ..docs import docstring
     18 from ..exceptions import ResourceLoadException
     19 from .action import ServiceAction, WaiterAction

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/docs/__init__.py:17
     13 import os
     15 from botocore.docs import DEPRECATED_SERVICE_NAMES
---> 17 from boto3.docs.service import ServiceDocumenter
     20 def generate_docs(root_dir, session):
     21     """Generates the reference documentation for botocore
     22 
     23     This will go through every available AWS service and output ReSTructured
   (...)
     30     :param session: The boto3 session
     31     """

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/docs/service.py:21
     19 import boto3
     20 from boto3.docs.client import Boto3ClientDocumenter
---> 21 from boto3.docs.resource import ResourceDocumenter, ServiceResourceDocumenter
     22 from boto3.utils import ServiceContext
     25 class ServiceDocumenter(BaseServiceDocumenter):
     26     # The path used to find examples

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/docs/resource.py:19
     16 from botocore.docs.bcdoc.restdoc import DocumentStructure
     17 from botocore.docs.utils import get_official_service_name
---> 19 from boto3.docs.action import ActionDocumenter
     20 from boto3.docs.attr import (
     21     document_attribute,
     22     document_identifier,
     23     document_reference,
     24 )
     25 from boto3.docs.base import BaseDocumenter

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/docs/action.py:26
     24 from boto3.docs.base import NestedDocumenter
     25 from boto3.docs.method import document_model_driven_resource_method
---> 26 from boto3.docs.utils import (
     27     add_resource_type_overview,
     28     get_resource_ignore_params,
     29     get_resource_public_actions,
     30 )
     32 PUT_DATA_WARNING_MESSAGE = """
     33 .. warning::
     34     It is recommended to use the :py:meth:`put_metric_data`
   (...)
     38     the metric resource's ``name`` attribute.
     39 """
     41 WARNING_MESSAGES = {
     42     "Metric": {"put_data": PUT_DATA_WARNING_MESSAGE},
     43 }

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/boto3/docs/utils.py:16
     13 import inspect
     15 import jmespath
---> 16 from botocore.docs.utils import DocumentModifiedShape  # noqa: F401
     19 def get_resource_ignore_params(params):
     20     """Helper method to determine which parameters to ignore for actions
     21 
     22     :returns: A list of the parameter names that does not need to be
     23         included in a resource's method call for documentation purposes.
     24     """

ImportError: cannot import name 'DocumentModifiedShape' from 'botocore.docs.utils' (/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/docs/utils.py)

In [3]:
# To Mitigate the aboce issue
%pip install --upgrade --force-reinstall boto3 botocore

  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 282.2 MB/s  0:00:00
Using cached jmespath-1.1.0-py3-none-any.whl (20 kB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.7.0
    Uninstalling urllib3-2.7.0:
      Successfully uninstalled urllib3-2.7.0
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [six]
  Attempting uninstall: jmespath━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [six]
    Found 

In [5]:
%pip list

Package                   Version
------------------------- -----------
aiobotocore               3.7.0
aiohappyeyeballs          2.7.1
aiohttp                   3.14.1
aioitertools              0.13.0
aiosignal                 1.4.0
annotated-doc             0.0.4
annotated-types           0.7.0
antlr4-python3-runtime    4.9.3
anyio                     4.14.1
asttokens                 3.0.1
async-timeout             5.0.1
attrs                     25.4.0
awscli                    1.45.36
boto3                     1.43.46
botocore                  1.43.46
certifi                   2026.6.17
cftime                    1.6.5
charset-normalizer        3.4.7
click                     8.4.2
cloudpickle               3.1.2
colorama                  0.4.6
comm                      0.2.3
debugpy                   1.8.21
decorator                 5.3.1
dill                      0.4.1
docker                    7.1.0
docutils                  0.19
exceptiongroup            1.3.1
executing         

In [6]:
import os
import time
import io
import re
import urllib.request

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

ImportError: cannot import name 'DocumentModifiedShape' from 'botocore.docs.utils' (/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/docs/utils.py)

In [18]:
%pip list | grep s3fs

s3fs                      2026.6.0
Note: you may need to restart the kernel to use updated packages.


## Cell 3 — Configuration
Edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [19]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/GEDI02_A/002/"     
GEDI_COUNTY_S3_PREFIX = "gedi-county-summary/"
GEDI-INFO_KEY         = "data/gedi-folder/city-of-charlottesville/gedi02_A_forCityOfCharlottesville.txt"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_multiyear-cville.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_grid-cville.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_summary-cville.csv")
OUTPUT_DETAILED_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_height-cville.csv")
OUTPUT_COUNTY_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_canopy_height-cville.csv")

# ── Spatial bounds (City of Charlottesville jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -78.5237, 38.0096, -78.4463, 38.0705

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Buckingham",      "51", "029",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI source  : s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project/gedi-county-summary/


## Cell 4 — Helper Functions

In [20]:
def parse_year_from_filename(filename: str):
    """Extract the year from a standard GEDI filename (e.g., GEDI02_A_2022143...)."""
    year_match = re.search(r'GEDI02_A_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None


def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from the US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    print(f"  Fetching boundary for {name}...")
    with urllib.request.urlopen(url, timeout=20) as r:
        gdf = gpd.read_file(r)
    if gdf.empty:
        raise ValueError(f"No boundary found for {name} (FIPS {state_fips}/{geo_id})")
    gdf = gdf.set_crs("EPSG:4326")
    gdf['jurisdiction'] = name
    return gdf


print("Helper functions defined.")

Helper functions defined.


## Cell 5 — Phase 1 & 1a: Discover GEDI Files in S3

In [1]:
def extract_h5_filename(url: str) -> str:
    """
    Extract just the .h5 file name (basename) from a full granule URL.
 
    Parameters
    ----------
    url : str
        Full URL to a .h5 granule, e.g.
        https://.../GEDI02_A_2025189013841_..._V002/GEDI02_A_2025189013841_..._V002.h5
 
    Returns
    -------
    str
        Just the filename, e.g. GEDI02_A_2025189013841_..._V002.h5
    """
    path = urlparse(url).path
    return posixpath.basename(path)
 
 
def read_url_list_from_s3(bucket: str, key: str, profile: str = None) -> list[str]:
    """
    Fetch a text object from S3 and return the .h5 filename from each non-empty line.
 
    Parameters
    ----------
    bucket : str
        S3 bucket name.
    key : str
        S3 object key (path within the bucket).
    profile : str, optional
        Named AWS CLI profile to use. If None, uses the default credential chain.
 
    Returns
    -------
    list[str]
        List of .h5 file names, one per non-blank line in the source file.
    """
    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    s3 = session.client("s3")
 
    try:
        log.info("Fetching s3://%s/%s ...", bucket, key)
        response = s3.get_object(Bucket=bucket, Key=key)
        body = response["Body"].read().decode("utf-8")
    except NoCredentialsError:
        log.error("No AWS credentials found. Configure ~/.aws/credentials, "
                   "set AWS_PROFILE, or pass --profile.")
        raise
    except ClientError as e:
        error_code = e.response.get("Error", {}).get("Code", "Unknown")
        if error_code == "NoSuchKey":
            log.error("Object not found: s3://%s/%s", bucket, key)
        elif error_code in ("AccessDenied", "403"):
            log.error("Access denied reading s3://%s/%s — check bucket permissions "
                       "or IAM role/profile.", bucket, key)
        else:
            log.error("S3 error (%s) reading s3://%s/%s: %s", error_code, bucket, key, e)
        raise
 
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    filenames = [extract_h5_filename(line) for line in lines]
    log.info("Read %d line(s) from s3://%s/%s and extracted %d .h5 filename(s)",
              len(lines), bucket, key, len(filenames))
    return filenames


In [ ]:
    
filenames = read_url_list_from_s3(S3_BUCKET, GEDI-INFO_KEY, profile=None)

print(f"\n{len(filenames)} GEDI .h5 file name(s):\n")

for name in filenames:
    print(name)


In [21]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI HDF5 files...")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5"):
            h5_keys.append(obj["Key"])

print(f"Found {len(h5_keys)} GEDI HDF5 files.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

Scanning s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/ for GEDI HDF5 files...
Found 390 GEDI HDF5 files.
  First : GEDI/GEDI02_A/002/GEDI02_A_2019113174925_O02048_03_T02621_02_003_01_V002.h5
  Last  : GEDI/GEDI02_A/002/GEDI02_A_2025189013841_O37209_02_T08829_02_004_02_V002.h5


## Cell 6 — Phase 1 & 1a: Batch Extraction with Spatial Masking and Quality Filtering

In [22]:
fs = s3fs.S3FileSystem(anon=False)
all_dfs = []

# This is a long running processing....let's time it
cell_start_time = time.time()

for i, key in enumerate(h5_keys):
    file_start_time = time.time()
    file_name = os.path.basename(key)

    # Phase 1a: Parse year from filename
    year = parse_year_from_filename(file_name)
    if not year:
        print(f"Warning: Could not parse year from {file_name}. Skipping.")
        continue

    try:
        # Stream HDF5 file from S3 into memory
        s3_path = f"s3://{S3_BUCKET}/{key}"
        with fs.open(s3_path, "rb") as f:
            raw_bytes = f.read()

        with h5py.File(io.BytesIO(raw_bytes), 'r') as hf:
            beams = [k for k in hf.keys() if k.startswith('BEAM')]

            for beam in beams:
                if 'lat_lowestmode' not in hf[beam]:
                    continue

                # Step 1: Extract coordinates first for rapid spatial masking
                lats = hf[f"{beam}/lat_lowestmode"][:]
                lons = hf[f"{beam}/lon_lowestmode"][:]

                spatial_mask = (
                    (lons >= MIN_LON) & (lons <= MAX_LON) &
                    (lats >= MIN_LAT) & (lats <= MAX_LAT)
                )

                if not np.any(spatial_mask):
                    continue

                # Step 2: Extract attributes only for points inside the bbox
                quality    = hf[f"{beam}/quality_flag"][:][spatial_mask]
                sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                rh98       = hf[f"{beam}/rh"][:, 98][spatial_mask]

                beam_df = pd.DataFrame({
                    'longitude':    lons[spatial_mask],
                    'latitude':     lats[spatial_mask],
                    'quality_flag': quality,
                    'sensitivity':  sensitivity,
                    'rh98':         rh98,
                    'year':         year,
                    'file_source':  file_name,
                    'beam':         beam
                })

                # Step 3: Strict quality filtering
                valid_df = beam_df[
                    (beam_df['quality_flag'] == 1) &
                    (beam_df['sensitivity']  > 0.9) &
                    (beam_df['rh98']         > 0)   &
                    (beam_df['rh98']         < 100)
                ]

                if not valid_df.empty:
                    all_dfs.append(valid_df)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    if (i + 1) % 10 == 0:
        file_elapsed = time.time() - file_start_time
        print(f"Processed {i + 1}/{len(h5_keys)} files... in {file_elapsed:.2f} seconds.")

#Calculate and display total run time
total_elapsed = time.time() - cell_start_time
minutes, seconds = divmod(total_elapsed, 60)

print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")


Processed 10/390 files... in 32.95 seconds.
Processed 20/390 files... in 33.42 seconds.
Processed 30/390 files... in 41.27 seconds.
Processed 40/390 files... in 38.00 seconds.
Processed 50/390 files... in 14.82 seconds.
Processed 60/390 files... in 43.57 seconds.
Processed 70/390 files... in 21.31 seconds.
Processed 80/390 files... in 26.22 seconds.
Processed 90/390 files... in 21.96 seconds.
Processed 100/390 files... in 26.13 seconds.
Processed 110/390 files... in 45.99 seconds.
Processed 120/390 files... in 64.10 seconds.
Processed 130/390 files... in 17.39 seconds.
Processed 140/390 files... in 23.84 seconds.
Processed 150/390 files... in 40.83 seconds.
Processed 160/390 files... in 32.52 seconds.
Processed 170/390 files... in 24.95 seconds.
Processed 180/390 files... in 21.65 seconds.
Processed 190/390 files... in 47.32 seconds.
Processed 200/390 files... in 15.77 seconds.
Processed 210/390 files... in 26.88 seconds.
Processed 220/390 files... in 21.52 seconds.
Processed 230/390 f

## Cell 7 — Save Extracted Points to Parquet

In [7]:
if not all_dfs:
    raise RuntimeError("No valid GEDI shots found within the bounding box. Check S3 prefix and bbox settings.")

df_gedi_cville = pd.concat(all_dfs, ignore_index=True)
df_gedi_cville.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Total valid GEDI shots saved : {len(df_gedi_cville):,}")
print(f"Years covered                : {sorted(df_gedi_cville['year'].unique())}")
print(f"Parquet file                 : {OUTPUT_PARQUET}")
df_gedi_cville.head()

Total valid GEDI shots saved : 2,059,963
Years covered                : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Parquet file                 : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_multiyear.parquet


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam
0,-79.171967,38.044788,1,0.900244,2.92,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
1,-79.171469,38.044455,1,0.916953,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
2,-79.170971,38.044123,1,0.918692,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
3,-79.170475,38.043790,1,0.912793,20.48,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
4,-79.169977,38.043457,1,0.907017,29.40,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000


In [8]:
#Save to CSV locally to support Canopy Height Change
df_gedi_cville.to_csv(OUTPUT_DETAILED_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CSV}")

#df_gedi_cville.to_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height.csv', index=False)
#print(f"Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height.csv")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height.csv


## Cell 8 — Phase 1b: Harmonize to the SMAP 9 km Grid

In [9]:
lon_bins = np.arange(MIN_LON, MAX_LON, GRID_RES)
lat_bins = np.arange(MIN_LAT, MAX_LAT, GRID_RES)

df_gedi_cville['lon_grid'] = pd.cut(df_gedi_cville['longitude'], bins=lon_bins, labels=lon_bins[:-1]).astype(float)
df_gedi_cville['lat_grid'] = pd.cut(df_gedi_cville['latitude'],  bins=lat_bins, labels=lat_bins[:-1]).astype(float)

gedi_grid = df_gedi_cville.groupby(['year', 'lat_grid', 'lon_grid'])['rh98'].mean().reset_index()

ds_gedi = gedi_grid.set_index(['year', 'lat_grid', 'lon_grid']).to_xarray()
ds_gedi.to_netcdf(OUTPUT_NETCDF)

print(f"Grid cells produced : {len(gedi_grid):,}")
print(f"NetCDF saved to     : {OUTPUT_NETCDF}")
gedi_grid.head()

Grid cells produced : 1,610
NetCDF saved to     : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_grid.nc


,year,lat_grid,lon_grid,rh98
0,2019,37.3296,-79.1721,16.509153
1,2019,37.3296,-79.0911,16.554823
2,2019,37.3296,-79.0101,13.830095
3,2019,37.3296,-78.9291,12.423847
4,2019,37.3296,-78.8481,13.842674


## Cell 9 — Phase 1c: Fetch Jurisdiction Boundaries from US Census TIGERweb

In [10]:
boundary_gdfs = []

for name, st_fips, geo_id, geo_type in TARGET_JURISDICTIONS:
    try:
        if name == "Charlottesville": 
            b_gdf = fetch_boundary(name, st_fips, geo_id, geo_type)
            boundary_gdfs.append(b_gdf)
    except Exception as e:
        print(f"  Failed to fetch boundary for {name}: {e}")

if not boundary_gdfs:
    raise RuntimeError("No jurisdiction boundaries fetched. Cannot perform spatial join.")

filtered_counties = pd.concat(boundary_gdfs, ignore_index=True)
print(f"\nBoundaries fetched: {len(filtered_counties)} jurisdiction(s)")
filtered_counties[['jurisdiction', 'geometry']].head(10)

  Fetching boundary for Albemarle...
  Fetching boundary for Augusta...
  Fetching boundary for Buckingham...
  Fetching boundary for Charlottesville...
  Fetching boundary for Fluvanna...
  Fetching boundary for Greene...
  Fetching boundary for Louisa...
  Fetching boundary for Nelson...
  Fetching boundary for Rockingham...

Boundaries fetched: 9 jurisdiction(s)


,jurisdiction,geometry
0,Albemarle,"POLYGON ((-78.64831 37.73272, -78.64824 37.732..."
1,Augusta,"POLYGON ((-78.93069 38.31342, -78.93242 38.314..."
2,Buckingham,"POLYGON ((-78.24812 37.64089, -78.248 37.64158..."
3,Charlottesville,"POLYGON ((-78.52368 38.02233, -78.52371 38.022..."
4,Fluvanna,"POLYGON ((-78.15905 37.74933, -78.15889 37.749..."
5,Greene,"POLYGON ((-78.3587 38.29019, -78.35879 38.2904..."
6,Louisa,"POLYGON ((-78.30676 38.00647, -78.30687 38.006..."
7,Nelson,"POLYGON ((-78.90797 37.97698, -78.90798 37.976..."
8,Rockingham,"POLYGON ((-79.09295 38.66441, -79.09295 38.664..."


## Cell 10 — Phase 1c: Spatial Join and County-Level Aggregation

In [11]:
gdf_gedi_cville = gpd.GeoDataFrame(
    df_gedi_cville,
    geometry=gpd.points_from_xy(df_gedi_cville.longitude, df_gedi_cville.latitude),
    crs="EPSG:4326"
)

print("Performing spatial join...")
gedi_with_county_cville = gpd.sjoin(
    gdf_gedi_cville,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

gedi_county_summary_cville = (
    gedi_with_county_cville
    .groupby(['year', 'jurisdiction'])['rh98']
    .mean()
    .reset_index()
    .rename(columns={'rh98': 'canopy_height_mean_m'})
)

print(f"Spatial join complete. Rows in summary: {len(gedi_county_summary_cville)}")
gedi_county_summary_cville

Performing spatial join...
Spatial join complete. Rows in summary: 61


,year,jurisdiction,canopy_height_mean_m
0,2019,Albemarle,19.675812
1,2019,Augusta,14.742975
2,2019,Buckingham,16.031614
3,2019,Charlottesville,16.583229
4,2019,Fluvanna,18.198132
...,...,...,...
56,2025,Fluvanna,15.802814
57,2025,Greene,14.136559
58,2025,Louisa,15.712695
59,2025,Nelson,17.517622


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[14], line 11
      4 gdf_gedi_canopy_height = gpd.GeoDataFrame(
      5      df_gedi_canopy_height,
      6      geometry=gpd.points_from_xy(df_gedi_canopy_height.longitude, df_gedi_canopy_height.latitude),
      7      crs="EPSG:4326"
      8  )
     10 print("Performing spatial join...")
---> 11 gedi_with_county_canopy_height = gpd.sjoin(
     12     gdf_gedi_canopy_height,
     13     filtered_counties[['jurisdiction', 'geometry']],
     14     how='inner',
     15     predicate='within'
     16 )
     18 # gedi_county_summary_cville = (
     19 #     gedi_with_county
     20 #     .groupby(['year', 'jurisdiction'])['rh98']
   (...)
     23 #     .rename(columns={'rh98': 'canopy_height_mean_m'})
     24 # )
     26 print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_height)}")

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/geopandas/tools/sjoin.py:119, in sjoin(left_df, right_df, how, predicate, lsuffix, rsuffix, distance, on_attribute, **kwargs)
    113 _basic_checks(left_df, right_df, how, lsuffix, rsuffix, on_attribute=on_attribute)
    115 indices = _geom_predicate_query(
    116     left_df, right_df, predicate, distance, on_attribute=on_attribute
    117 )
--> 119 joined, _ = _frame_join(
    120     left_df,
    121     right_df,
    122     indices,
    123     None,
    124     how,
    125     lsuffix,
    126     rsuffix,
    127     predicate,
    128     on_attribute=on_attribute,
    129 )
    131 return joined

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/geopandas/tools/sjoin.py:461, in _frame_join(left_df, right_df, indices, distances, how, lsuffix, rsuffix, predicate, on_attribute)
    459 right_nlevels = right_df.index.nlevels
    460 right_index_original = right_df.index.names
--> 461 right_df, right_column_names = _reset_index_with_suffix(right_df, rsuffix, left_df)
    463 # if conflicting names in left and right, add suffix
    464 left_column_names, right_column_names = _process_column_names_with_suffix(
    465     left_column_names,
    466     right_column_names,
   (...)
    469     right_df,
    470 )

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/geopandas/tools/sjoin.py:280, in _reset_index_with_suffix(df, suffix, other)
    278         # check new label will not be in other dataframe
    279         if new_label in df.columns or new_label in other.columns:
--> 280             raise ValueError(
    281                 f"'{new_label}' cannot be a column name in the frames being joined"
    282             )
    283         column_names[i] = new_label
    284 return df_reset, pd.Index(column_names)

ValueError: 'index_right' cannot be a column name in the frames being joined

In [15]:
# Load dataframe with canopy height data points
df_gedi_cville_canopy_height = pd.read_csv(OUTPUT_COUNTY_CANOPY_HEIGHT_CSV)

gdf_gedi_cville_canopy_height = gpd.GeoDataFrame(
     df_gedi_cville_canopy_height,
     geometry=gpd.points_from_xy(df_gedi_cville_canopy_height.longitude, df_gedi_cville_canopy_height.latitude),
     crs="EPSG:4326"
 )

# The Fix for the Value Error

# Drop any stale index columns from a previous join
gdf_gedi_cville_canopy_height = gdf_gedi_cville_canopy_height.drop(
    columns=[c for c in ["index_right", "index_left"] if c in gdf_gedi_cville_canopy_height.columns]
)
filtered_counties = filtered_counties.drop(
    columns=[c for c in ["index_right", "index_left"] if c in filtered_counties.columns]
)

print("Performing spatial join...")
gedi_with_county_canopy_height = gpd.sjoin(
    gdf_gedi_cville_canopy_height,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

# gedi_county_summary_cville = (
#     gedi_with_county
#     .groupby(['year', 'jurisdiction'])['rh98']
#     .mean()
#     .reset_index()
#     .rename(columns={'rh98': 'canopy_height_mean_m'})
# )

print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_height)}")
gedi_with_county_canopy_height.head(20)

Performing spatial join...
Spatial join complete. Rows in summary: 993671


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam,geometry,jurisdiction_left,index_right,jurisdiction_right
0,-79.171967,38.044788,1,0.900244,2.92,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17197 38.04479),Augusta,1,Augusta
1,-79.171469,38.044455,1,0.916953,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17147 38.04446),Augusta,1,Augusta
2,-79.170971,38.044123,1,0.918692,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17097 38.04412),Augusta,1,Augusta
3,-79.170475,38.043790,1,0.912793,20.48,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17047 38.04379),Augusta,1,Augusta
4,-79.169977,38.043457,1,0.907017,29.40,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16998 38.04346),Augusta,1,Augusta
5,-79.169478,38.043124,1,0.961559,28.69,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16948 38.04312),Augusta,1,Augusta
6,-79.168979,38.042791,1,0.961136,7.79,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16898 38.04279),Augusta,1,Augusta
7,-79.168481,38.042459,1,0.915506,25.69,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16848 38.04246),Augusta,1,Augusta
8,-79.167985,38.042126,1,0.909027,24.83,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16798 38.04213),Augusta,1,Augusta
9,-79.167488,38.041793,1,0.921234,3.07,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16749 38.04179),Augusta,1,Augusta


In [16]:
#Save to CSV to support County Canopy Height Change
gedi_with_county_canopy_height.to_csv(OUTPUT_COUNTY_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CANOPY_HEIGHT_CSV}")
#gedi_with_county_canopy_height.to_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv', index=False)
#print(f"Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv


## Cell 11 — Save County Summary Locally and Upload to S3

In [17]:
# Save locally
gedi_county_summary_cville.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_COUNTY_CSV)
s3_county_key       = GEDI_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_summary.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary.csv

All phases completed successfully!


## Cell 12 — Upload County to S3